# LG-CXR FRCNN Kaggle Notebook

Use this notebook inside Kaggle. It clones the GitHub repository, uses the competition dataset from `/kaggle/input/amia-public-challenge-2026`, and writes all outputs to `/kaggle/working`.

## 1. Clone Or Update Project

Internet must be enabled in the Kaggle notebook settings for this cell. If you imported this notebook directly from the GitHub repo, this still keeps `/kaggle/working/Amia_keagle_challenge` synchronized.

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/thomas240103/Amia_keagle_challenge.git'
PROJECT_DIR = Path('/kaggle/working/Amia_keagle_challenge')

if PROJECT_DIR.exists():
    print('Project already exists, pulling latest main...')
    !git -C /kaggle/working/Amia_keagle_challenge pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', Path.cwd())

## 2. Install Lightweight Dependencies

Kaggle already includes PyTorch, torchvision, pandas, numpy, Pillow, tqdm, and matplotlib in most GPU images. This cell installs only likely-missing extras.

In [ ]:
!pip install -q PyYAML timm

## 3. Check Environment

In [ ]:
import platform
import torch
import torchvision

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    x = torch.ones((2, 2), device='cuda')
    print('CUDA tensor test:', x.sum().item())

## 4. Configure Kaggle Paths

Before running this cell, add the AMIA Public Challenge 2026 dataset to the Kaggle notebook using the right-side **Add input** panel.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = '/kaggle/input/amia-public-challenge-2026'
WORK_DIR = '/kaggle/working'

os.environ['LGCXR_DATA_ROOT'] = DATA_ROOT
os.environ['LGCXR_WORK_DIR'] = WORK_DIR
os.environ['LGCXR_ALLOW_WEIGHT_DOWNLOAD'] = '1'  # Use pretrained torchvision detector weights when Kaggle internet is enabled.

print('DATA_ROOT =', DATA_ROOT, 'exists:', Path(DATA_ROOT).exists())
print('WORK_DIR =', WORK_DIR, 'exists:', Path(WORK_DIR).exists())
print('Expected files/folders:')
for name in ['train', 'test', 'train.csv', 'test.csv', 'img_size.csv', 'sample_submission.csv']:
    p = Path(DATA_ROOT) / name
    print(f'  {name}:', p.exists())

## 5. Local Repository Checks

In [ ]:
!python scripts/06_ci_checks.py

## 6. Preflight

Run this before any training. It validates dataset discovery, images, annotation columns, class 14 handling, and original-to-PNG box scaling.

In [ ]:
!python scripts/00_preflight.py --config configs/baseline_frcnn.yaml

## 7. Dimension Audit

This checks original dimensions from `img_size.csv`, actual PNG dimensions, and how boxes scale into PNG/model space.

In [ ]:
!python scripts/05_audit_dimensions.py --config configs/baseline_frcnn.yaml

## 8. Smoke Test

This runs a small one-epoch pipeline and creates a format-valid `submission.csv`. Do this before full training.

In [ ]:
!python scripts/04_full_pipeline.py --config configs/baseline_frcnn.yaml --smoke-test

## 9. Full Training

Run this after preflight, dimension audit, and smoke test are clean.

In [ ]:
!python scripts/01_train_scanner.py --config configs/baseline_frcnn.yaml

## 10. Inspect V1 Scanner Metrics

Use these metrics before deciding whether to continue to V2. The most important value is `best_score`, which is the validation VOC-style mAP proxy at IoU 0.4.

In [ ]:
import json
from pathlib import Path

summary_path = Path('/kaggle/working/lgcxr_scanner_training_summary.json')
print('summary exists:', summary_path.exists())
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('best_score:', summary.get('best_score'))
    for row in summary.get('history', [])[-5:]:
        print({k: row.get(k) for k in ['epoch', 'train_loss', 'val_map_proxy']})
    last_metrics = summary.get('history', [{}])[-1].get('metrics', {})
    print('last validation mAP proxy:', last_metrics.get('map'))
    print('last per-class AP:', last_metrics.get('per_class_ap'))

## 11. Inference

In [ ]:
!python scripts/02_predict_scanner.py --config configs/baseline_frcnn.yaml

## 12. Make Submission

In [ ]:
!python scripts/03_make_submission.py --config configs/baseline_frcnn.yaml

## 13. Inspect Submission

In [ ]:
import pandas as pd
from pathlib import Path

submission_path = Path('/kaggle/working/submission.csv')
print('submission exists:', submission_path.exists())
if submission_path.exists():
    sub = pd.read_csv(submission_path)
    print(sub.shape)
    display(sub.head())

## 14. Optional Full Pipeline Command

Use this instead of separate training/inference/submission cells when you want one command.

In [ ]:
# !python scripts/04_full_pipeline.py --config configs/baseline_frcnn.yaml --force-train